# AI-WAF Model Training Pipeline (Google Colab / Local)

This notebook orchestrates the training, calibration, and evaluation for the **Next-Gen AI-WAF** models:
1. **Model 1**: TF-IDF Character n-grams (2-5) + Calibrated Logistic Regression
2. **Model 2**: LightGBM on Structural / Domain Features
3. **Ensemble & Export**: Generates joblib artifacts with SHA-256 checksums.

## 1. Environment Setup

In [ ]:
!pip install -r requirements-training.txt
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
import lightgbm

print(f"scikit-learn version: {sklearn.__version__}")
print(f"LightGBM version: {lightgbm.__version__}")

## 2. Clean & Preprocess Data

In [ ]:
# Run data cleaning pipeline
!python -m training.clean_data --raw-dir data/raw --output-dir data/cleaned
# Perform stratified train/validation/test split
!python -m training.prepare_features --input data/cleaned/all_data.csv --output-dir data/splits

## 3. Train Model 1: TF-IDF + Logistic Regression

In [ ]:
# Trains vectorizer.pkl, model_lr.pkl, and calibrator.pkl
!python -m training.train_lr --splits-dir data/splits --output-dir trained_models

## 4. Train Model 2 (Optional): LightGBM on Structural Features

In [ ]:
# Trains model_lgbm.pkl if time permits
!python -m training.train_lgbm --splits-dir data/splits --output-dir trained_models

## 5. Evaluation & Reporting

In [ ]:
# Runs evaluation against reserved test set and outputs classification report & confusion matrix
!python -m training.evaluate --splits-dir data/splits --models-dir trained_models --output-dir docs